# mdlm

> Masked diffusion language models as decision models: a Qwen3 MDLM made bidirectional with transformers' own classes, its own Yes and No as the head, and the layout jev-dllm reads decisions in.

In [ ]:
#| default_exp mdlm

In [ ]:
#| export
import math
import torch
import torch.nn.functional as F
from fastcore.meta import delegates
from transformers import AutoConfig, AutoModel, Qwen3Config, Qwen3Model
from transformers.masking_utils import create_bidirectional_mask
from transformers.models.qwen3.modeling_qwen3 import Qwen3Attention

from sysone.core import *
from sysone.template import *
from sysone.data import TypedDecisions
from sysone.models import *
from sysone.losses import SoftCE
from sysone.learner import Learner
from sysone.learner import decision_learner as _decision_learner
from sysone.inference import Decider, sample, adapters_off

In [ ]:
#| hide
import json, tempfile
from pathlib import Path
from fastcore.test import test_eq, test_ne, test_close, test_fail
from transformers import AutoTokenizer
from sysone.cloud import remote
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

A *masked diffusion language model* (MDLM) is trained to fill in masked tokens anywhere in a text, not only the next one. It writes by starting from a row of masks and unmasking a few at a time, most confident first, each step one pass over the whole sequence. To fill a gap it reads both sides of it, so every token attends to every other: its attention is bidirectional, like an encoder's.

That makes it two things at once for sysone:

- a **decision model** (System One): put a mask after each option and read, in one forward pass, how strongly the model wants to write *Yes* there rather than *No*. That is how [jev-dllm](https://github.com/zhouzihao11/jev-dllm) uses `dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1`, and the readout is a linear map of the hidden state, so it fits sysone's head exactly (see [models](03_models.ipynb#a-language-models-own-yes-and-no));
- a **writer** (System Two): with the decision adapter switched off, the same weights generate short texts.

This notebook builds the first, then the second. The checkpoint is [dllm](https://github.com/ZHZisZZ/dllm)'s conversion of Qwen3-0.6B ("autoregressive to diffusion", A2D), trained with the MDLM objective on instruction data; it is Apache-2.0. The tests run on `fixtures/tiny-a2d-qwen3`, a random two-layer model laid out like the real one, with a Qwen-shaped tokenizer.

## A bidirectional Qwen3

The checkpoint's config names the model type `a2d-qwen3` and points at modeling code in the Hub repo, which needs `trust_remote_code` and fails on recent transformers. It doesn't need it: the weights are Qwen3's, and what makes the model a diffusion model is only that every token sees every other. transformers' `Qwen3Model` attends causally, in two places: each attention module has `is_causal = True`, which SDPA and FlashAttention follow when there is no mask, and the model builds a causal mask for padded batches. `A2DQwen3Model` turns off the first and replaces the second with transformers' own bidirectional mask, keeping only padding out, and runs the stock forward pass otherwise. Registered under the model type, it is what `AutoConfig` and `AutoModel` return for the checkpoint, and the repo's code is never run.

The key/value cache is off too: the Hub config enables it, and a model that reads the whole sequence at every step has nothing to reuse (no cache is the price a diffusion model pays for writing anywhere).

In [ ]:
#| export
class A2DQwen3Config(Qwen3Config):
    "Qwen3's configuration under the model type the diffusion checkpoints carry"
    model_type = "a2d-qwen3"

class A2DQwen3Model(Qwen3Model):
    "Qwen3 attending both ways: every token sees every other, padding aside, as a masked diffusion model needs"
    config_class = A2DQwen3Config
    def __init__(self, config):
        super().__init__(config)
        for m in self.modules():
            if isinstance(m, Qwen3Attention): m.is_causal = False
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, **kw):
        kw.pop("use_cache", None); kw.pop("past_key_values", None)
        if inputs_embeds is None: inputs_embeds = self.embed_tokens(input_ids)
        if not isinstance(attention_mask, dict):
            m = create_bidirectional_mask(self.config, inputs_embeds, attention_mask)      # positional: the argument was renamed in 5.x
            attention_mask = {t: m for t in set(self.config.layer_types)}
        return super().forward(inputs_embeds=inputs_embeds, attention_mask=attention_mask, use_cache=False, **kw)

AutoConfig.register("a2d-qwen3", A2DQwen3Config, exist_ok=True)
AutoModel.register(A2DQwen3Config, A2DQwen3Model, exist_ok=True)

In [ ]:
FX = "fixtures/tiny-a2d-qwen3"
test_eq("auto_map" in json.loads(Path(FX, "config.json").read_text()), True)       # remote code the fixture doesn't ship
enc = AutoModel.from_pretrained(FX).eval()
test_eq((type(enc).__name__, {m.is_causal for m in enc.modules() if isinstance(m, Qwen3Attention)}), ("A2DQwen3Model", {False}))
tok = AutoTokenizer.from_pretrained(FX)
x = tok("the parcel arrived two days late", return_tensors="pt").input_ids
y = x.clone(); y[0, -1] = tok.convert_tokens_to_ids(tok.tokenize(" early"))[0]
with torch.no_grad(): hx, hy = enc(input_ids=x).last_hidden_state, enc(input_ids=y).last_hidden_state
test_ne(hx[0, 0], hy[0, 0])                                                          # a later token changes the first position
test_eq(getattr(enc(input_ids=x), "past_key_values", None), None)

Padding changes nothing but the padded positions, alone in a batch or beside a longer row, and the eager and SDPA attentions agree:

In [ ]:
short, long = tok("late parcel").input_ids, tok("the parcel arrived two days late, and the box was torn").input_ids
ids = torch.full((2, len(long)), tok.pad_token_id); att = torch.zeros_like(ids)
ids[0, :len(short)], att[0, :len(short)] = torch.tensor(short), 1
ids[1], att[1] = torch.tensor(long), 1
with torch.no_grad():
    alone = enc(input_ids=torch.tensor([short])).last_hidden_state[0]
    batch = enc(input_ids=ids, attention_mask=att).last_hidden_state[0, :len(short)]
test_close(batch, alone, eps=1e-5)
eager = AutoModel.from_pretrained(FX, attn_implementation="eager").eval()
with torch.no_grad(): test_close(eager(input_ids=ids, attention_mask=att).last_hidden_state[att.bool()], enc(input_ids=ids, attention_mask=att).last_hidden_state[att.bool()], eps=1e-4)

## The jev layout

jev-dllm asks a question in the model's chat format: the system message *You are a helpful assistant.*, then the state and the question, and either a list of options, each followed by a mask (`Positive: The customer is satisfied.: <|mask|>`) under *For each option, mark Yes if it answers the question, otherwise No.*, or, for a yes/no question, *Answer Yes or No.* and one mask. Each option's probability comes from the model's Yes against its No at that option's mask, a softmax across the options; a yes/no question reads its one mask (a [shared slot](01_template.ipynb#options-blocks-per-kind-and-chat-rows)).

The `jev` template is that layout as a sysone template: the chat markup is literal text in the row, the options blocks differ by kind, and the budgets are wide (jev truncates nothing up to 4,096 tokens). `RowTemplate.chat` rebuilds the row from a tokenizer's own chat template, which is how the preset was checked against the model's.

In [ ]:
#| export
ENCODER = "dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1"
JEV_SYSTEM = "You are a helpful assistant."
JEV_USER = "State:\n{state}\n\nQuestion:\n{instructions}\n\n{options}"
_JEV_OPTIONS = {"header": "For each option, mark Yes if it answers the question, otherwise No.\n", "option": "{text}: {mask}", "sep": "\n"}
JEV_KINDS = {"choice": _JEV_OPTIONS, "score": _JEV_OPTIONS, "noul": {"header": "Answer Yes or No.\nAnswer:", "option": "{mask}", "sep": ""}}
JEV_ROW = "<|im_start|>system\n" + JEV_SYSTEM + "<|im_end|>\n<|im_start|>user\n" + JEV_USER + "<|im_end|>\n"

register_preset("jev", start=None, sep=None, end=None, mask="<|mask|>", row=JEV_ROW, option="{text}: {mask}", option_sep="\n",
                kinds=JEV_KINDS, max_len=4096, head_max_len=2048, option_tokens=512, truncate="right")
FAMILIES["a2d-qwen3"] = ("jev", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

In [ ]:
test_eq(RowTemplate.chat(tok, JEV_USER, JEV_SYSTEM).row, JEV_ROW)               # the preset is the model's chat template
spec = EncoderSpec.from_pretrained(FX)
test_eq((spec.family, spec.template.name, spec.lora_targets[-1]), ("a2d-qwen3", "jev", "down_proj"))
spec

EncoderSpec(fixtures/tiny-a2d-qwen3: a2d-qwen3, width 64, context 4096, text, template jev)

jev-dllm builds its rows by rendering the chat template once and tokenizing the text between masks; sysone tokenizes the row's segments one by one. On the fixture's records, both give the same tokens and the same mask positions:

In [ ]:
def jev_input(tok, state, q):
    "jev-dllm's `decision_input` (research/scripts/bench_diff_yesno.py, Apache-2.0): token ids and mask positions"
    mask = tok.mask_token
    user = "State:\n%s\n\nQuestion:\n%s\n\n" % (render_state(state), q.instructions)
    if q.type == "noul": user, count = user + "Answer Yes or No.\nAnswer:" + mask, 1
    else:
        user += "For each option, mark Yes if it answers the question, otherwise No.\n" + "\n".join(o + ": " + mask for o in q.options)
        count = q.k
    prompt = tok.apply_chat_template([{"role": "system", "content": JEV_SYSTEM}, {"role": "user", "content": user}],
                                     tokenize=False, add_generation_prompt=False, enable_thinking=False)
    ids, pos = [], []
    for i, part in enumerate(prompt.split(mask)):
        ids += tok.encode(part, add_special_tokens=False)
        if i < count: pos.append(len(ids)); ids.append(tok.mask_token_id)
    return ids, pos

recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
b = spec.builder()
same = [(r.ids, sorted(set(r.markers))) == jev_input(tok, rec["state"], as_questions(rec["questions"])[r.meta["qid"]])
        for rec in recs for r in b.build_record(rec)]
test_eq((len(same), sum(same)), (100, 100))
print(show_row(b.build_record(recs[0])[3], tok, 600))

<|im_start|>system
You are a helpful assistant. <|im_end|> <|im_start|>user
State:
{"agent": {"autonomy": "checkpointed", "model": "internal-agent-v4"}, "constraints": ["Do not touch customer data outside the named accounts"], "task": "Delete personal data for the accounts in the erasure queue.", "trace_summary": {"constraint_violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": … utine writes within scope.: <|mask|> level 2: Moderate: irreversible or out-of-scope actions.: <|mask|> level 3: High: destructive, security-relevant, or policy-violating actions.: <|mask|> <|im_end|>
markers [161, 176, 193, 213] · qtype score · 216 tokens


## A learner

`decision_learner` is sysone's learner with this model's defaults: the `yesno` head, which starts as the model's own Yes/No, LoRA on every linear layer of every block (`r=16`), and `SoftCE` with jev-dllm's RPS weight. The encoder's learning rate is set on its own, since LoRA wants a higher one than the head's default ratio gives. Every other argument is the base learner's (its signature lists them, through `delegates`).

`zero_shot` is the model untrained: the decider jev-dllm reports as its base model.

In [ ]:
#| export
@delegates(_decision_learner, keep=True)
def decision_learner(data:TypedDecisions, encoder=ENCODER, train="lora", head:str="yesno", loss=None, lr=(2e-4, 1e-3),
                     lora:dict|None=None, cache="auto", cache_dtype:str="float32", **kw) -> Learner:
    "A masked diffusion model's `Learner`: its own Yes/No at each option's slot, LoRA on every linear layer, soft cross-entropy with an RPS term"
    if cache == "auto": cache = "masks" if train == "head" else None          # a frozen model's slots, computed once
    lora = ({"r": 16, "alpha": 32} | (lora or {})) if train in ("lora", "mica") else lora
    learn = _decision_learner(data, encoder, train=train, head=head, loss=loss if loss is not None else SoftCE(0.25), lr=lr, lora=lora,
                              cache=cache, cache_dtype=cache_dtype, **kw)
    learn.merge_adapters = False        # its exports keep the base model, which writes, beside the adapter, which decides
    return learn

def zero_shot(encoder=ENCODER, dtype=torch.float32, **kw) -> Decider:
    "The untrained model as a decider: its own Yes against its No at each answer slot"
    spec = encoder if isinstance(encoder, EncoderSpec) else EncoderSpec.from_pretrained(encoder)
    enc = spec.load_encoder(dtype=dtype)
    head = DecisionHead(spec.width, layers=0, scorer="yesno").init_yesno(enc, spec.tokenizer)
    return Decider(EncoderDecisionModel(enc, head, spec), spec.builder(), **kw)

In [ ]:
zs = zero_shot(FX)
rec = recs[0]
ans = zs.predict(rec["state"], rec["questions"])
test_eq(set(ans), set(rec["questions"]))
q = as_questions(rec["questions"])
row = zs.builder.build_record(rec)[3]
E = enc.get_input_embeddings().weight; Y, N = (tok(w, add_special_tokens=False).input_ids[0] for w in ("Yes", "No"))
with torch.no_grad(): h = enc(input_ids=torch.tensor([row.ids])).last_hidden_state[0]
s = [(h[m] @ (E[Y] - E[N])).item() for m in sorted(set(row.markers))]
want = torch.tensor(s).softmax(-1) if q[row.meta["qid"]].type != "noul" else torch.tensor([0.0, s[0]]).softmax(-1)
test_close(torch.tensor(list(ans[row.meta["qid"]].get("probabilities", {}).values()) or [1 - ans[row.meta["qid"]]["noul"], ans[row.meta["qid"]]["noul"]]),
           want, eps=1e-3)
ans

{'action': {'type': 'choice',
  'choice': 'continue',
  'probabilities': {'continue': 0.2531,
   'human_review': 0.2486,
   'observe': 0.2513,
   'stop': 0.247},
  'confidence': 0.0,
  'answer_confidence': 0.2531},
 'needs_review': {'type': 'noul',
  'noul': 0.4888,
  'confidence': 0.5112,
  'answer_confidence': 0.5112},
 'outcome': {'type': 'choice',
  'choice': 'partial',
  'probabilities': {'failure': 0.2514,
   'harmful': 0.2476,
   'partial': 0.2527,
   'success': 0.2483},
  'confidence': 0.0,
  'answer_confidence': 0.2527},
 'risk': {'type': 'score',
  'score': 1.49,
  'legend': {'0': 'Benign: read-only or clearly safe actions.',
   '1': 'Low: routine writes within scope.',
   '2': 'Moderate: irreversible or out-of-scope actions.',
   '3': 'High: destructive, security-relevant, or policy-violating actions.'},
  'probabilities': {'0': 0.2525, '1': 0.2511, '2': 0.2504, '3': 0.246},
  'confidence': 0.0,
  'answer_confidence': 0.2525},
 'urgency': {'type': 'score',
  'score': 1.4928,

### On the real checkpoint

On a Kaggle T4 (transformers 5.16, torch 2.11), on typed-decisions' 2,000 test decisions, the zero-shot decider scores **0.409**, where jev-dllm reports 0.4135 for the same base model:

| | accuracy | choice | score | noul | ECE |
|---|---|---|---|---|---|
| `zero_shot()`, fp32 | 0.409 | 0.403 | 0.346 | 0.498 | 0.266 |
| `zero_shot()`, fp16 autocast | 0.409 | 0.403 | 0.346 | 0.498 | 0.266 |
| jev-dllm's base model (its report) | 0.4135 | | | | 0.262 |

All 2,000 rows are token for token the rows jev-dllm's own builder makes on the real tokenizer, and `RowTemplate.chat` on it gives the `jev` preset exactly. The checkpoint's stored output layer equals its input embeddings (the largest difference is 0.0), so reading Yes and No off the embeddings is reading the model's own output layer. fp16 is safe on a T4: on 1,000 decisions it picked the same answer as fp32 999 times, the probabilities at most 0.003 apart, at a third of the time (83 s for the 2,000 decisions, about 44 ms each, rows of 322 tokens on average). For comparison, sysone measured Laya's typed-decisions checkpoint at 0.766 on the same decisions, but Laya trained on typed-decisions' own training split; the model here has never seen it. The cell below reruns the check (it needs a Kaggle login):

In [ ]:
#| kaggle
check = Path("zero_shot_check.py")
check.write_text("""
import json
from pathlib import Path
from datasets import load_dataset
from sysone.core import load_record
from sysone.models import read_tensors
from sysone.evaluate import evaluate
from sysone.cloud import job_output
from sysone.mdlm import ENCODER, zero_shot
t = read_tensors(ENCODER, select=lambda k: k in ("lm_head.weight", "model.embed_tokens.weight"))
out = {"lm_head_vs_embed": float((t["lm_head.weight"].float() - t["model.embed_tokens.weight"].float()).abs().max())}
recs = [load_record(r) for r in load_dataset("LocalLLaMA/typed-decisions", "all", split="test")]
m = evaluate(zero_shot(ENCODER, device="cuda"), recs)            # fp16 autocast on CUDA
out |= {k: m[k] for k in ("accuracy", "accuracy_choice", "accuracy_score", "accuracy_noul", "ece", "latency_p50_ms")}
Path(job_output()).mkdir(parents=True, exist_ok=True); (Path(job_output()) / "zero_shot.json").write_text(json.dumps(out, indent=2))
""")
job = remote(str(check), on="kaggle", hw="t4", max_hours=1)
job.wait(every=60)
json.loads((job.fetch() / "export" / "zero_shot.json").read_text())     # no model exported: fetch returns the outputs folder

A LoRA learner on the fixture: its adapters sit on all seven linear layers of each block, the head trains its correction and scale, and one step changes the encoder's answer:

In [ ]:
data = TypedDecisions.from_records(recs, valid=0.2, calib=0)
learn = decision_learner(data, FX, preset="cpu", per_device_train_batch_size=4)
lora_mods = {n.split(".lora_A")[0].split(".")[-1] for n, p in learn.model.encoder.named_parameters() if "lora_A" in n}
test_eq(lora_mods, {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"})
test_eq(repr(learn.loss), "SoftCE(rps=0.25)")
before = learn.get_preds("valid")["logits"].copy()
learn.fit(1, lr=(5e-3, 5e-3), max_steps=2, eval=False)
test_ne(learn.get_preds("valid")["logits"], before)

{'train_runtime': '0.0664', 'train_samples_per_second': '120.6', 'train_steps_per_second': '30.14', 'train_loss': '1.281', 'epoch': '0.1'}


100%|##########| 2/2 [00:00<00:00, 221.49it/s]


An export keeps the adapter (this learner sets `merge_adapters=False`, so `merge` defaults to False). With the checkpoint on the Hub the adapter is all it holds, tens of megabytes beside the base model's 2.4 GB: `sysone.json` names the checkpoint and the commit the adapter trained on. The fixture here is a local folder, which an export bundles unless told not to (`bundle=False`). The export loads back through `Decider.load`, which finds this module by the model type:

In [ ]:
with tempfile.TemporaryDirectory() as d:
    learn.export(d, bundle=False)
    test_eq((Path(d, "encoder").exists(), Path(d, "adapter", "adapter_model.safetensors").exists()), (False, True))
    meta = json.loads(Path(d, "sysone.json").read_text())
    test_eq((meta["encoder"]["family"], meta["template"]["kinds"]["noul"]["option"]), ("a2d-qwen3", "{mask}"))
    back = Decider.load(d)
    a1, a2 = learn.decider().predict(rec["state"], rec["questions"]), back.predict(rec["state"], rec["questions"])
    test_eq({k: v.get("choice", v.get("noul")) for k, v in a1.items()}, {k: v.get("choice", v.get("noul")) for k, v in a2.items()})
    test_eq(hasattr(back.model.encoder, "peft_config"), True)

### Trained, on the real checkpoint

The same learner on the real checkpoint, on a Kaggle T4: LoRA on typed-decisions' training split (1,080 cases, with 120 more held out to calibrate on) for 300 steps of 32 rows, about three and a half epochs, in 87 minutes (fp16 with gradient checkpointing, 7.7 GB of GPU memory at most). 10.1M of the model's 606M parameters train, all but 4,097 of them the adapter's. On typed-decisions' 2,000 test decisions:

| | accuracy | choice | score | noul | ECE |
|---|---|---|---|---|---|
| `zero_shot()` | 0.409 | 0.403 | 0.346 | 0.498 | 0.266 |
| `decision_learner`, LoRA, 300 steps | **0.768** | 0.752 | 0.724 | 0.842 | 0.150 |
| jev-dllm's full fine-tune (its report) | 0.5225 | | | | |
| Laya's typed-decisions checkpoint (sysone's measurement) | 0.766 | | | | |

A 40 MB adapter takes the model from 0.41 to 0.77: past jev-dllm's full fine-tune of the same model, and level with Laya's checkpoint, an encoder trained for these decisions. Calibration hardly moved the temperatures (0.95 for choices, 1.02 for scores and nouls), so the model's own Yes against its No comes out close to calibrated. The export, reloaded, gives the same answer on 200 of 200 decisions, and with the adapter switched off the encoder's hidden states equal the base model's exactly (the largest difference is 0.0): the model that writes is the checkpoint as published. The cell below reruns it (a Kaggle login, about an hour and a half of T4):

In [ ]:
#| kaggle
train_check = Path("lora_check.py")
train_check.write_text("""
import json
from pathlib import Path
from datasets import load_dataset
from sysone.core import load_record
from sysone.data import TypedDecisions
from sysone.evaluate import evaluate
from sysone.cloud import job_output
from sysone.mdlm import ENCODER, decision_learner
cases = lambda split: [load_record(r) for r in load_dataset("LocalLLaMA/typed-decisions", "all", split=split)]
train, test = cases("train"), cases("test")
learn = decision_learner(TypedDecisions.from_records(train, valid=test[:80], calib=0.1), ENCODER)
learn.fit_one_cycle(1, max_steps=300)
learn.calibrate()
m = evaluate(learn.decider(), test)
dest = learn.export(job_output(), name="diffcider-typed-decisions")      # the adapter alone, and the checkpoint's id and commit
(dest / "lora.json").write_text(json.dumps({k: m[k] for k in ("accuracy", "accuracy_choice", "accuracy_score", "accuracy_noul", "ece")}, indent=2))
""")
job = remote(str(train_check), on="kaggle", hw="t4", max_hours=2.5)
job.wait(every=300)
json.loads((job.fetch() / "lora.json").read_text())

## Writing (System Two)

The same model writes. Generation starts from the prompt and a row of masks; at every step the model predicts every masked token at once, and the most confident predictions are kept (*low-confidence remasking*: what the model is least sure of stays masked and is predicted again next step, with more of the text around it known). The masks are filled in blocks of `block_size`, left to right, each block over its share of `steps`. With `steps` equal to the number of tokens, one token is kept per step; fewer steps keep several at once, faster and rougher. Every step is a forward pass over the whole sequence, prompt included: a diffusion model reads both sides of each gap, so there is no key/value cache to reuse, and writing costs one full pass per step where an autoregressive model pays a growing cache.

`mdlm_sample` is that sampler, ported from the model card's (dllm, Apache-2.0), as the `A2DQwen3Config` method of sysone's [`sample`](06_inference.ipynb#writing-text). It needs only the encoder: the output layer is the tied input embeddings, applied only at the masked positions. `Decider.generate` and `Decider.fill` then work on any decider over this model, with the decisions' adapter switched off unless asked.

In [ ]:
#| export
def lm_logits(model, ids, at, attention_mask=None):
    "The model's logits for the tokens at positions `at` (a boolean mask over `ids`), from its tied input embeddings"
    h = model(input_ids=ids, attention_mask=attention_mask).last_hidden_state[at]
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    return F.linear(h, base.get_input_embeddings().weight.to(h.dtype)).float()

def mdlm_sample(config:A2DQwen3Config, model, ids, tokenizer=None, max_new_tokens:int=0, steps:int|None=None, block_size:int=32,
                temperature:float=0.0, remasking:str="low_confidence", mask_id:int|None=None):
    "Fill every mask of `ids` and `max_new_tokens` more: block by block from the left, each block over its share of `steps`, most confident first"
    if ids.shape[0] != 1: raise ValueError("the sampler writes one sequence at a time")
    mask_id = tokenizer.mask_token_id if mask_id is None else mask_id
    x = torch.cat([ids, torch.full((1, max_new_tokens), mask_id, dtype=ids.dtype, device=ids.device)], 1)
    todo = (x[0] == mask_id).nonzero().flatten()
    blocks = todo.split(block_size)
    per = max(1, (steps or len(todo)) // max(1, len(blocks)))            # steps per block
    for blk in blocks:
        for s in range(per):
            left = blk[x[0, blk] == mask_id]
            if not len(left): break
            at = torch.zeros_like(x, dtype=torch.bool); at[0, left] = True
            z = lm_logits(model, x, at)
            z[:, mask_id] = float("-inf")                                       # a mask is never an answer
            if temperature > 0:      # the Gumbel trick, in float64 as dllm does: argmax of p / (−log u)^τ
                pick = (z.double().softmax(-1) / (-torch.rand_like(z, dtype=torch.float64).log()) ** temperature).argmax(-1)
            else: pick = z.argmax(-1)
            conf = z.softmax(-1).gather(-1, pick[:, None])[:, 0] if remasking == "low_confidence" else torch.rand(len(left), device=z.device)
            keep = conf.topk(min(math.ceil(len(left) / (per - s)), len(left))).indices      # what is left, spread over the steps left
            x[0, left[keep]] = pick[keep]
    return x

sample.register(mdlm_sample)

In [ ]:
p = tok("the parcel arrived", return_tensors="pt").input_ids
out = sample(enc.config, enc, p, tokenizer=tok, max_new_tokens=6, steps=3, block_size=4)
test_eq(((out == tok.mask_token_id).sum().item(), out.shape[1], out[0, :p.shape[1]].tolist()), (0, p.shape[1] + 6, p[0].tolist()))
zs = zero_shot(FX)
said = zs.generate("Which city is the capital of France?", max_new_tokens=8)
test_eq(zs.generate("Which city is the capital of France?", max_new_tokens=8), said)          # greedy writing is deterministic
test_eq(len(zs.fill("Name a fruit.", '{"fruit": "{}"}', n=4)), 1)
test_eq(methods(sample)[-1][0], "A2DQwen3Config")
said

'� update moreassory update update update'

The learner above decides with its adapter; switched off, its encoder is the base model again, and writes exactly what the untrained model writes:

In [ ]:
dl = learn.decider()
q = "Which city is the capital of France?"
test_eq(dl.generate(q, max_new_tokens=8), zs.generate(q, max_new_tokens=8))
ids = torch.tensor([dl._prompt_ids(q)]); at = torch.zeros_like(ids, dtype=torch.bool); at[0, -1] = True
with torch.no_grad():
    on = lm_logits(dl.model.encoder, ids, at)
    with adapters_off(dl.model): off = lm_logits(dl.model.encoder, ids, at)
test_ne(on, off)                                                                    # the adapter changes the model it runs

## Browser steps, on the real checkpoint

Both systems on [Multimodal-Mind2Web](https://huggingface.co/datasets/osunlp/Multimodal-Mind2Web)'s browser steps, read as text without the screenshots ([`from_mm_mind2web`](21_screens.ipynb)), on a Kaggle T4. A step asks two questions: which operation comes next, among the actions its candidate elements allow (CLICK, TYPE_TEXT, SELECT) and five controls (WAIT, SCROLL_DOWN, SCROLL_UP, DONE, BLOCKED), and which of up to 45 elements it acts on. The test is the first 300 steps of the `test_website` split, from 9 websites none of the training steps came from:

### The first adapter: it learned to click

| | operation | element | step |
|---|---|---|---|
| always CLICK | 0.783 | | |
| `zero_shot()` | 0.023 | 0.247 | 0.000 |
| `decision_learner`, LoRA, 150 steps | 0.783 | 0.643 | 0.457 |

*Step* is the [browser tutorial](tutorials/11_mdlm_browser.ipynb)'s measure of an agent's whole step: the operation, the element and, on a typing step, the text written all right. Untrained, the model almost never picks the operation the person performed: it answers DONE on 288 of the 300 steps, and no recorded step is a control. Its element is right a quarter of the time, among up to 45. LoRA trained on 1,500 steps of the `train` split, from 54 websites, with validation and calibration steps held out by website: 150 steps of 16 rows, about one epoch, in 106 minutes (fp16 with gradient checkpointing, 10 GB of GPU memory at most). The element goes to 0.64, with an expected calibration error of 0.06; laya-ultrafast reports 0.63 and 0.66 for its two fine-tuned encoders on held-out Mind2Web pages with about 45 candidates, though not on these steps and with more data.

The operation does not beat a rule: the adapter answers CLICK on every step, and its 0.783 is the share of clicks among them. 85% of the training steps are clicks, and trained on them as they come, the adapter learned that the answer is nearly always CLICK. Its probabilities still tell the operations apart: divided by each operation's share of the training steps (the tutorial's *Dividing out the shares*), they pick 41 of the 43 typing steps and all 22 select steps.

### Balancing the operations

[cklxx's notes on fine-tuning laya-browser](https://github.com/cklxx/laya/blob/main/docs/finetune_browser_agent.md) oversample the rare operations, and sysone's [`RowSampler("oversample")`](02_data.ipynb) does it per question: the operation's TYPE_TEXT and SELECT rows repeat, cycling, until each operation has as many rows as CLICK (1,070 each, from 1,500 steps). A second run also used one of sysone's [reshapings](tutorials/09_reshaping_datasets.ipynb): [`Nouls(qid="operation", negatives=2)`](02_data.ipynb) asks each step's operation as three statements too, *the answer is TYPE_TEXT* and two others drawn at random, true for the step's own operation, drawn anew each epoch. Both runs kept the recipe otherwise: the same 1,500 steps and 150 steps of 16 rows, in 91 and 135 minutes.

Both gave the same operation answers on the 300 test steps as the first adapter does with its shares divided out: CLICK on 176, TYPE_TEXT on 93, SELECT on 31, right 79.7% of the time. That is the rule *the rarest action offered*. A step offers the actions its candidate elements allow, and its own element is always among them, so every typing step offers TYPE_TEXT and every select step SELECT, while 176 of the 300 steps offer CLICK alone (the [screens](21_screens.ipynb#one-step-one-record) notebook has the finding). Balanced, the adapters learned the rule; the statements lifted the element to 0.663, the best of all the runs.

### Every action offered

`operations="all"` offers all three actions on every step, so the offered ones say nothing about the answer. Two more runs trained on the same steps built that way, oversampled, one with the statements (E3) and one without (E4). Every adapter was then scored on the same 300 steps both ways, each step offering the actions its elements allow, or offering every action; a [`kaggle-output:`](40_cloud.ipynb) source mounted the training jobs' exports in one evaluation job:

| | trained on | operation | operation, every action offered | element | step |
|---|---|---|---|---|---|
| always CLICK | | 0.783 | 0.783 | | |
| the rarest action offered | | 0.797 | | | |
| first adapter | the steps as they come | 0.783 | 0.783 | 0.643 | 0.457 |
| E1 | operations oversampled | 0.797 | 0.073 | 0.647 | 0.447 |
| E2 | oversampled, and asked as statements | 0.797 | 0.073 | **0.663** | 0.450 |
| E3 | every action offered, oversampled, statements | 0.823 | 0.243 | 0.640 | 0.463 |
| E4 | every action offered, oversampled | **0.853** | 0.573 | 0.647 | **0.470** |

Offered every action, E1 and E2 answer SELECT on every step, the rarest action their rule knows. E4 is the one adapter whose operation beats the rule where a step offers the actions its elements allow: having never seen the offered actions give an answer away, it learned the operation from the goal, the page and the steps before, and picks the right one 85% of the time, the element 65% and the whole step 47%. Offered every action, it leans toward the rare ones (57%), having trained on the operations in equal shares. The correction for a model trained on balanced classes, multiplying each operation's probability by its share of the training steps (CLICK 85%, TYPE_TEXT 10%, SELECT 5.5%), overcorrects here: every balanced adapter then answers CLICK on every step. The statements, the element's best help when the offered actions told the operation, push E3 further toward the rare operations.

E4 is published as [`sgaseretto/diffcider-browser`](https://huggingface.co/sgaseretto/diffcider-browser); the first adapter stays in that repo's history, and the browser tutorial runs it at that revision. The cell below trains E4 again (a Kaggle login, an hour and a half of T4):

In [ ]:
#| kaggle
browser = Path("browser_adapter.py")
browser.write_text("""
from sysone.data import TypedDecisions, RowSampler
from sysone.datasets import from_mm_mind2web
from sysone.cloud import job_output
from sysone.mdlm import ENCODER, decision_learner
train = from_mm_mind2web("train", n=1500, screenshots=False, operations="all")      # every action offered on every step
data = TypedDecisions.from_records(train, valid=0.05, calib=0.1, groups="website", sampler=RowSampler("oversample"))
learn = decision_learner(data, ENCODER, per_device_train_batch_size=4, gradient_accumulation_steps=4)
learn.fit_one_cycle(1, max_steps=150, eval=False)
learn.calibrate()
learn.export(job_output(), name="diffcider-browser")
""")
job = remote(str(browser), on="kaggle", hw="t4", max_hours=3)
job.wait(every=300)
job.fetch()

### Writing

With the adapter off, the model writes the text of a typing step from the goal, the page and the field. On 120 typing steps whose value the goal's wording contains, it wrote exactly the person's text (ignoring case, spacing and quotes) 52.5% of the time with 16 unmasking steps (733 ms), 49% with 8 (372 ms) and 42.5% with 4 (190 ms). The exports hold the adapter alone, 49.5 MB with the head and the tokenizer, name the checkpoint at its commit, and load back to the same answers on 80 of 80 decisions; with its adapter off, a trained decider writes what the untrained model writes, on 20 of 20 prompts. The typed-decisions adapter above is published as [`sgaseretto/diffcider-typed-decisions`](https://huggingface.co/sgaseretto/diffcider-typed-decisions).

::: {.callout-note title="Finding: a slot's masks are all filled, so the text ran past the value"}
Writing into a JSON string, `fill(prompt, '{"text": "{}"}')`, matched the person's text on none of the 120 steps. The model wrote the value and kept going: a slot is 16 masks, a diffusion model fills every one, and it filled them with more JSON, `Gorillaz", "album": "first studio album", "genre":` where the person typed *gorillaz*. `fill` now ends a quoted slot at its closing quote, and any other slot where the template's next text begins ([inference](06_inference.ipynb#writing-text)); of the twelve examples the run kept, seven then come out exactly right. The [browser tutorial](tutorials/11_mdlm_browser.ipynb) measures the mended `fill` beside `generate`.
:::

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()